In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, RandomSampler, TensorDataset
import pickle
import re
import unicodedata
import random
import numpy as np
from compress_fasttext.models import CompressedFastTextKeyedVectors
import torchtext; torchtext.disable_torchtext_deprecation_warning()
from torchtext.data import get_tokenizer
from torch.distributions.categorical import Categorical
from jiwer import wer  # pip install jiwer

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Special tokens
SOS_token = 0
EOS_token = 1
PAD_token = 2

# Hyperparameters
MAX_LENGTH = 5
embedding_dim = 100
hidden_size = 512
# For RL training, we use a batch size of 1 (for simplicity)
batch_size = 1  
dropout_p = 0.2
learning_rate = 0.001

# Load embeddings
embedder = CompressedFastTextKeyedVectors.load('cc.de.100.reduced.bin')

# ---------------------------
# Helper Functions
# ---------------------------
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \\1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

def tokenize(sentence):
    tokenizer = get_tokenizer("moses", language='de')
    return tokenizer(sentence)

def clean_token(token):
    token = token.lower()
    token = re.sub(r"[^a-zA-Z0-9]+", "", token)
    return token

def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence if word in lang.word2index]

# ---------------------------
# Language Class
# ---------------------------
class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.index2word = {SOS_token: "SOS", EOS_token: "EOS", PAD_token: "PAD"}
        self.n_words = 3  # Count SOS, EOS, and PAD

    def addSentence(self, sentence):
        for word in sentence:
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

# ---------------------------
# Data Preparation
# ---------------------------
def readLangs(file, reverse=False):
    with open(file, 'rb') as f:
        data = pickle.load(f)
    pairs = [[tokenize(de), [clean_token(token) for token in dgs]] 
             for de, dgs in zip(data['de'], data['dgs'])]
    if reverse:
        input_lang = Lang('dgs')
        output_lang = Lang('de')
    else:
        input_lang = Lang('de')
        output_lang = Lang('dgs')
    return input_lang, output_lang, pairs

def filterPair(p):
    return len(p[0]) < MAX_LENGTH and len(p[1]) < MAX_LENGTH

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def prepareData(file, reverse=False):
    input_lang, output_lang, pairs = readLangs(file, reverse)
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    pairs = filterPairs(pairs)
    return input_lang, output_lang, pairs

# ---------------------------
# Sentence Embedding & Data Loader
# ---------------------------
def sentence2vec(tokens):
    embeddings = [torch.tensor(embedder[token], device=device) if token in embedder 
                  else torch.zeros(embedding_dim, device=device) 
                  for token in tokens]
    return torch.stack(embeddings).squeeze()

def get_dataloader(translations_file, batch_size):
    input_lang, output_lang, pairs = prepareData(translations_file, False)
    n = len(pairs)
    input_vecs = torch.zeros((n, MAX_LENGTH, embedding_dim))
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    input_sentences = []  # Original input sentences (strings)
    target_sentences = []  # Original target sentences (strings)
    for idx, (inp, tgt) in enumerate(pairs):
        tgt_ids = indexesFromSentence(output_lang, tgt)
        tgt_ids.append(EOS_token)
        inp_vec = sentence2vec(inp)
        while inp_vec.dim() < 2:
            inp_vec = inp_vec.unsqueeze(0)
        input_vecs[idx, :inp_vec.size(0), :] = inp_vec[:, :]
        target_ids[idx, :len(tgt_ids)] = tgt_ids
        input_sentences.append(' '.join(inp))
        target_sentences.append(' '.join(tgt))
    train_data = TensorDataset(input_vecs.to(device),
                               torch.LongTensor(target_ids).to(device))
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader, input_sentences, target_sentences

# ---------------------------
# Model Architecture
# ---------------------------
# Encoder
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        # input_size equals embedding_dim (using precomputed embeddings)
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)
    def forward(self, input_tensor):
        output, hidden = self.gru(self.dropout(input_tensor))
        return output, hidden

# Bahdanau Attention
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)
    def forward(self, query, keys, mask):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys))).squeeze(-1)
        scores.masked_fill_(~mask, -float('inf'))
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), keys)
        return context, weights

# Decoder with Attention and step() for RL sampling
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size, padding_idx=PAD_token)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)
    def forward(self, encoder_outputs, encoder_hidden, target_tensor, mask):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.full((batch_size, 1), SOS_token, dtype=torch.long, device=device)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        for t in range(MAX_LENGTH):
            embedded = self.dropout(self.embedding(decoder_input))
            query = decoder_hidden.permute(1, 0, 2)
            context, _ = self.attention(query, encoder_outputs, mask)
            gru_input = torch.cat((embedded, context), dim=2)
            output, decoder_hidden = self.gru(gru_input, decoder_hidden)
            output = self.out(output.squeeze(1))
            # Always append the output for later use:
            decoder_outputs.append(output)
            
            # Teacher forcing: use ground truth with some probability
            if target_tensor is not None and random.random() < 0.5:
                decoder_input = target_tensor[:, t].unsqueeze(1)
            else:
                decoder_input = output.argmax(dim=1).unsqueeze(1)
        return torch.stack(decoder_outputs, dim=1)

            # For supervised training, teacher forcing can be applied (here we mix)\n            if random.random() < 0.5 and target_tensor is not None:\n                decoder_input = target_tensor[:, t].unsqueeze(1)\n            else:\n                decoder_input = output.argmax(dim=1).unsqueeze(1)\n        return torch.stack(decoder_outputs, dim=1)
    def step(self, encoder_outputs, decoder_hidden, input_token, mask):
        # Single decoding step for RL sampling
        embedded = self.dropout(self.embedding(input_token))
        query = decoder_hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, encoder_outputs, mask)
        gru_input = torch.cat((embedded, context), dim=2)
        output, decoder_hidden = self.gru(gru_input, decoder_hidden)
        output = self.out(output.squeeze(1))
        return output, decoder_hidden, attn_weights

# Seq2Seq model wrapper with RL sampling
class Seq2SeqWithRL(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    def forward(self, src, trg, mask):
        encoder_outputs, encoder_hidden = self.encoder(src)
        outputs = self.decoder(encoder_outputs, encoder_hidden, trg, mask)
        return outputs
    def sample(self, src):
        # For RL training, we want gradients so we do not use torch.no_grad()
        encoder_outputs, encoder_hidden = self.encoder(src)
        batch_size = src.size(0)  # Should be 1 for RL training if processing instance by instance
        mask = torch.ones((batch_size, encoder_outputs.size(1)), dtype=torch.bool, device=device)
        decoder_input = torch.full((batch_size, 1), SOS_token, dtype=torch.long, device=device)
        decoder_hidden = encoder_hidden
        outputs = []     # List of token IDs (each is a tensor of shape [batch]) 
        log_probs = []   # List of log probabilities (each is a tensor of shape [batch])
        for t in range(MAX_LENGTH):
            output, decoder_hidden, _ = self.decoder.step(encoder_outputs, decoder_hidden, decoder_input, mask)
            probs = F.softmax(output, dim=-1)  # [batch, output_size]
            dist = Categorical(probs)
            sampled_token = dist.sample()  # [batch] (for batch size 1, a single-element tensor)
            log_prob = dist.log_prob(sampled_token)  # [batch]
            outputs.append(sampled_token)  
            log_probs.append(log_prob)
            decoder_input = sampled_token.unsqueeze(1)
            if sampled_token.item() == EOS_token:
                break
        outputs = torch.stack(outputs, dim=0)  # [T, batch]
        log_probs = torch.stack(log_probs, dim=0)  # [T, batch]
        return outputs, log_probs

# ---------------------------
# Simple Tokenizer using output_lang vocabulary
# ---------------------------
class SimpleTokenizer:
    def __init__(self, lang):
        self.lang = lang
    def decode(self, token_ids):
        # token_ids can be a list of ints
        if isinstance(token_ids, list):
            return [self.lang.index2word.get(t, '<UNK>') for t in token_ids]
        else:
            return self.lang.index2word.get(token_ids, '<UNK>')
    def encode(self, sentence):
        tokens = sentence.split()
        return [self.lang.word2index.get(token, 0) for token in tokens]

# ---------------------------
# Reward Function using WER
# ---------------------------
def compute_reward(pred_tokens, target_tokens):
    # Convert lists of tokens into sentences
    pred_sentence = ' '.join(pred_tokens)
    target_sentence = ' '.join(target_tokens)
    # Reward: 1.0 - WER (so lower WER gives higher reward)
    return 1.0 - wer(target_sentence, pred_sentence)

# ---------------------------
# Reinforcement Learning Training Loop (REINFORCE)
# ---------------------------
def train_reinforce(model, dataloader, optimizer, tokenizer, device, gamma=0.99):
    model.train()
    total_loss = 0
    # Process one instance at a time (batch size = 1)
    for batch_idx, (src, trg) in enumerate(dataloader):
        # Use the first instance of the batch for RL training
        src_instance = src[0].unsqueeze(0)  # [1, seq_len, embedding_dim]
        trg_instance = trg[0].unsqueeze(0)  # [1, seq_len]
        optimizer.zero_grad()
        # Sample a translation using the RL sampling method
        outputs, log_probs = model.sample(src_instance)  # outputs: [T, 1], log_probs: [T, 1]
        # Decode sampled outputs to tokens
        outputs_list = outputs.squeeze(1).tolist()  # list of token ids
        pred_tokens = tokenizer.decode(outputs_list)
        # Decode target tokens (remove trailing zeros if any)
        target_indices = trg_instance.squeeze(0).tolist()
        target_tokens = tokenizer.decode(target_indices)
        reward = compute_reward(pred_tokens, target_tokens)
        # Policy gradient loss: multiply mean log probability by reward (maximize reward)
        policy_loss = -torch.mean(log_probs) * reward
        policy_loss.backward()
        optimizer.step()
        total_loss += policy_loss.item()
        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}, Reward: {reward:.4f}, Loss: {policy_loss.item():.4f}")
    return total_loss / len(dataloader)

# ---------------------------
# Evaluation Function (Supervised Evaluation for Reference)
# ---------------------------
def evaluate(n=5):
    model.eval()  # Set the model in evaluation mode
    examples = []
    with torch.no_grad():
        # Iterate over the evaluation dataloader (which might have a larger batch size)
        for batch_idx, (input_tensor, target_tensor) in enumerate(dataloader):
            batch_size = input_tensor.size(0)
            # Evaluate each instance individually using the sample() method
            for i in range(batch_size):
                src_instance = input_tensor[i].unsqueeze(0)  # Shape: [1, seq_len, embedding_dim]
                outputs, _ = model.sample(src_instance)  # Use the sample() method (which returns outputs and log_probs)
                # outputs is of shape [T, 1]; convert it to a list of token IDs
                predicted_indices = outputs.squeeze(1).tolist()  
                # Map token IDs to words, ignoring special tokens
                predicted_sentence = ' '.join(
                    output_lang.index2word[idx] for idx in predicted_indices if idx not in [SOS_token, EOS_token, PAD_token]
                )
                index = batch_idx * batch_size + i
                # Ensure we don't go out of bounds
                if index >= len(input_sentences) or index >= len(target_sentences):
                    continue
                input_sentence = input_sentences[index]
                target_sentence = target_sentences[index]
                examples.append((input_sentence, target_sentence, predicted_sentence))
                if len(examples) >= n:
                    break
            if len(examples) >= n:
                break
    # Print the examples
    for inp, tgt, pred in examples:
        print("Input:    ", inp)
        print("Target:   ", tgt)
        print("Predicted:", pred)
        print("-" * 50)



In [23]:
# ---------------------------
# Main Execution
# ---------------------------
file = "from_transcripts.pkl"

# Load data
input_lang, output_lang, dataloader, input_sentences, target_sentences = get_dataloader(file, batch_size)

# Initialize models
encoder = EncoderRNN(embedding_dim, hidden_size, dropout_p).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words, dropout_p).to(device)

# Load saved states if available; otherwise, train from scratch
try:
    encoder.load_state_dict(torch.load("encoder_sd_512_rl.pt"))
    decoder.load_state_dict(torch.load("decoder_sd_512_rl.pt"))
except Exception as e:
    print("No saved model states found; training from scratch.")

# Wrap encoder and decoder into the Seq2SeqWithRL model
model = Seq2SeqWithRL(encoder, decoder)

# Initialize a simple tokenizer using the output language
tokenizer = SimpleTokenizer(output_lang)

# Choose an optimizer
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


No saved model states found; training from scratch.


In [24]:

# Train using pure reinforcement learning (REINFORCE)
n_epochs_rl = 1  # Adjust the number of RL epochs as needed
for epoch in range(1, n_epochs_rl + 1):
    rl_loss = train_reinforce(model, dataloader, optimizer, tokenizer, device, gamma=0.99)
    print(f"RL Epoch {epoch}/{n_epochs_rl}, Loss: {rl_loss:.4f}")

# Save model states
torch.save(encoder.state_dict(), 'encoder_sd_512_rl.pt')
torch.save(decoder.state_dict(), 'decoder_sd_512_rl.pt')


Batch 0, Reward: 0.0000, Loss: 0.0000
Batch 10, Reward: 0.0000, Loss: 0.0000
Batch 20, Reward: 0.0000, Loss: 0.0000
Batch 30, Reward: 0.0000, Loss: 0.0000
Batch 40, Reward: 0.0000, Loss: 0.0000
Batch 50, Reward: 0.0000, Loss: 0.0000
Batch 60, Reward: 0.0000, Loss: 0.0000
Batch 70, Reward: 0.0000, Loss: 0.0000
Batch 80, Reward: 0.0000, Loss: 0.0000
Batch 90, Reward: 0.0000, Loss: 0.0000
Batch 100, Reward: 0.0000, Loss: 0.0000
Batch 110, Reward: 0.0000, Loss: 0.0000
Batch 120, Reward: 0.0000, Loss: 0.0000
Batch 130, Reward: 0.0000, Loss: 0.0000
Batch 140, Reward: 0.0000, Loss: 0.0000
Batch 150, Reward: 0.0000, Loss: 0.0000
Batch 160, Reward: 0.0000, Loss: 0.0000
Batch 170, Reward: 0.0000, Loss: 0.0000
Batch 180, Reward: 0.0000, Loss: 0.0000
Batch 190, Reward: 0.0000, Loss: 0.0000
Batch 200, Reward: 0.0000, Loss: 0.0000
Batch 210, Reward: 0.0000, Loss: 0.0000
Batch 220, Reward: 0.0000, Loss: 0.0000
Batch 230, Reward: 0.0000, Loss: 0.0000
Batch 240, Reward: 0.0000, Loss: 0.0000
Batch 250, 

In [26]:

# Evaluate the model (supervised evaluation for reference)
evaluate()


Input:     Entschuldigung .
Target:    entschuldigung1
Predicted: 
--------------------------------------------------
Input:     Ja , genau .
Target:    ja2
Predicted: 
--------------------------------------------------
Input:     Ich /
Target:    index1 ich1
Predicted: 
--------------------------------------------------
Input:     Was ?
Target:    was1b
Predicted: 
--------------------------------------------------
Input:     Aber was ?
Target:    oral
Predicted: 
--------------------------------------------------
